# 🌳 CART Decision Tree — Binary Classification of Wine Quality

Building, training, and evaluating a **CART Decision Tree** model to predict
whether a red wine is **high quality** (`is_high_quality = 1`, quality ≥ 7).

| | |
|---|---|
| **Target variable (Y)** | `is_high_quality` (1 = quality ≥ 7, 0 = quality < 7) |
| **Predictor variables (X)** | `alcohol`, `volatile acidity`, `sulphates`, `citric acid`, `fixed acidity`, `density`, `pH`, `chlorides`, `residual sugar`, `free sulfur dioxide`, `total sulfur dioxide` |
| **Model** | `DecisionTreeClassifier` — Gini criterion (**CART**, scikit-learn) |
| **Validation strategy** | Three splits: 70/30 · 40/60 · 80/20 (train + test) + optimal `max_depth` search |
| **Class imbalance handling** | Oversampling of the minority class with `sklearn.utils.resample` (train only) |

> **Dataset:** UCI Wine Quality — 1,599 red wine samples with physicochemical properties and a sensory score.
> Unlike Logistic Regression, the CART tree **does not require scaling** and measures the importance
> of each predictor through **Gini impurity reduction**.

## I. Initial Setup — Imports and Visual Style

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Consistent visual style across the project
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅ Libraries imported successfully")
print(f"   pandas   {pd.__version__}")
print(f"   numpy    {np.__version__}")
print(f"   seaborn  {sns.__version__}")
print("\n⚠️  Note: the CART tree does not require StandardScaler — X_raw is used directly.")

## II. Loading Data from UCI

We load the `winequality-red.csv` dataset and directly build the binary target `is_high_quality`
using a threshold on `quality`.

- **quality ≥ 7** → `is_high_quality = 1` (high quality wine)
- **quality < 7** → `is_high_quality = 0` (standard or low quality wine)

In [ ]:
# ── CELL 2: Load data from UCI ────────────────────────────────────────────────
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

wine_quality = pd.read_csv(URL, sep=";")

# Build binary target: quality >= 7 → high quality
wine_quality['is_high_quality'] = (wine_quality['quality'] >= 7).astype(int)

print(f"✅ Data loaded: {wine_quality.shape[0]:,} rows × {wine_quality.shape[1]} columns")
print(f"\nAvailable columns:\n{wine_quality.columns.tolist()}")
print(f"\nTarget variable balance:")
bal = wine_quality['is_high_quality'].value_counts()
pct = wine_quality['is_high_quality'].value_counts(normalize=True) * 100
print(f"   High quality  (quality >= 7) -> class 1: {bal[1]:,}  ({pct[1]:.1f}%)")
print(f"   Other quality (quality < 7) -> class 0: {bal[0]:,}  ({pct[0]:.1f}%)")
print(f"\nOriginal quality distribution:")
print(wine_quality['quality'].value_counts().sort_index())
wine_quality.head()

## III. Dataset Validation and Exploration

In [ ]:
# ── CELL 3: Validation and Exploration ────────────────────────────────────────
print("=" * 55)
print("DATASET VALIDATION")
print("=" * 55)

# Types and nulls
print("\n📋 Data types and missing values:")
print(wine_quality.dtypes.to_frame('dtype').join(
    wine_quality.isnull().sum().to_frame('nulls')
))

# Class balance
balance = wine_quality['is_high_quality'].value_counts()
pct     = wine_quality['is_high_quality'].value_counts(normalize=True) * 100
print(f"\n🎯 Target variable balance:")
print(f"   High quality  (1): {balance[1]:>5,}  ({pct[1]:.1f}%)")
print(f"   Other quality (0): {balance[0]:>5,}  ({pct[0]:.1f}%)")
print(f"\n⚠️  Imbalanced dataset — class 1 (high quality) is the minority.")
print(f"   Minority-class oversampling with resample() will be applied to TRAIN only.")

# Per-class statistics
print(f"\n📊 Means per class (high quality vs rest):")
features_num = ['alcohol', 'volatile acidity', 'sulphates', 'citric acid',
                'fixed acidity', 'density', 'pH']
print(wine_quality.groupby('is_high_quality')[features_num].mean().round(3).to_string())

## IV. Data Preprocessing

### Applied steps
1. **Feature definition** — all available physicochemical variables as predictors.
2. **Null check** — the UCI dataset contains no missing values.
3. **No scaling** — the CART tree is invariant to variable scaling.

> **Key difference vs Logistic Regression:** the CART tree **does not require scaling** (`StandardScaler`).
> Tree splits are based on value thresholds, which are invariant to scaling. `X_raw` is used directly as the feature matrix.

In [ ]:
# ── CELL 4: Preprocessing ─────────────────────────────────────────────────────
df = wine_quality.copy()

# Predictor variables: all physicochemical (drop quality and target)
drop_cols = ['quality', 'is_high_quality']
X = df.drop(columns=drop_cols)
y = df['is_high_quality']

print("✅ Preprocessing completed")
print(f"   Shape X : {X.shape}")
print(f"   Shape y : {y.shape}")
print(f"\n   Predictor columns ({len(X.columns)}):")
for col in X.columns:
    print(f"     · {col}")
print(f"\n   Classes in y -> 0: {(y==0).sum():,}  |  1: {(y==1).sum():,}")
print("\n⚠️  No StandardScaler applied — CART trees are invariant to scaling.")

## V. Minority-Class Oversampling with `resample`

The dataset is imbalanced (~14% high quality). An unconstrained CART tree on imbalanced data tends to
**overfit toward the majority class and ignore the minority class**. To counter this we **oversample the
minority class** in the training set with `sklearn.utils.resample` (sampling with replacement) until both
classes are balanced.

> **Key rule:** oversampling is applied **only to the training set**, after the train/test split.
> The test set keeps its real distribution so the evaluation is honest.

In [ ]:
# ── CELL 5: Oversampling helper (train only) ──────────────────────────────────
def oversample_minority(X_train, y_train, random_state=42):
    """
    Oversample the minority class in the TRAINING set only, using
    sklearn.utils.resample (sampling with replacement) until both
    classes are balanced 50/50. Returns (X_train_bal, y_train_bal).
    """
    train = X_train.copy()
    train['__target__'] = y_train.values

    majority = train[train['__target__'] == 0]
    minority = train[train['__target__'] == 1]

    minority_upsampled = resample(
        minority, replace=True,
        n_samples=len(majority), random_state=random_state
    )
    balanced = pd.concat([majority, minority_upsampled]).sample(
        frac=1, random_state=random_state
    )
    y_bal = balanced['__target__']
    X_bal = balanced.drop(columns='__target__')
    return X_bal, y_bal

print("✅ `oversample_minority` defined — applied to TRAIN only (resample, 50/50)")

## VI. Training and Evaluation — Three Splits (Train + Test)

For each split (70/30, 40/60, 80/20) the same process is followed:
1. Split `X` and `y` into train/test with `stratify=y`.
2. **Oversample the minority class on the training set only** with `resample`.
3. Train `DecisionTreeClassifier` (Gini criterion, **CART**) on the balanced training data.
4. Evaluate on **both** train and test.
5. Compute the 5 metrics: Accuracy, Precision, Recall, F1-Score, ROC-AUC.

In [ ]:
# ── CELL 6: Train + evaluate function (with oversampling) ─────────────────────
def train_evaluate(X, y, test_size, random_state=42, name="Split", max_depth=None):
    """
    Train a CART DecisionTreeClassifier on an oversampled training set and
    evaluate on train and test. Returns a dict with train/test metrics plus
    data for the ROC curve and confusion matrix.

    CART parameters:
      · criterion='gini'   — Gini impurity (standard CART)
      · max_depth          — None = unlimited tree (watch for overfitting)
    Instead of class_weight='balanced', minority-class OVERSAMPLING with
    sklearn.utils.resample is used (train only).
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Oversample minority class — TRAIN ONLY
    X_train_bal, y_train_bal = oversample_minority(X_train, y_train, random_state)

    model = DecisionTreeClassifier(
        criterion='gini',
        max_depth=max_depth,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=random_state,
    )
    model.fit(X_train_bal, y_train_bal)

    # Predictions on TRAIN (balanced) and TEST (original)
    y_pred_tr = model.predict(X_train_bal)
    y_prob_tr = model.predict_proba(X_train_bal)[:, 1]
    y_pred    = model.predict(X_test)
    y_prob    = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    return {
        'name'      : name,
        'train_size': len(X_train_bal),
        'test_size' : len(X_test),
        'train_acc' : accuracy_score(y_train_bal, y_pred_tr),
        'train_f1'  : f1_score(y_train_bal, y_pred_tr, zero_division=0),
        'train_auc' : roc_auc_score(y_train_bal, y_prob_tr),
        'accuracy'  : accuracy_score(y_test, y_pred),
        'precision' : precision_score(y_test, y_pred, zero_division=0),
        'recall'    : recall_score(y_test, y_pred, zero_division=0),
        'f1'        : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'   : roc_auc_score(y_test, y_prob),
        'fpr'       : fpr,
        'tpr'       : tpr,
        'y_test'    : y_test,
        'y_pred'    : y_pred,
        'model'     : model,
    }

print("✅ `train_evaluate` defined")
print("   · CART DecisionTreeClassifier — Gini criterion")
print("   · Trained on oversampled (resample) training set; train+test evaluated")

In [ ]:
# ── CELL 7: Run the three splits ──────────────────────────────────────────────
results = [
    train_evaluate(X, y, test_size=0.30, name="Split 70/30"),
    train_evaluate(X, y, test_size=0.60, name="Split 40/60"),
    train_evaluate(X, y, test_size=0.20, name="Split 80/20"),
]

# ── Print metrics to console ──────────────────────────────────────────────────
print("=" * 78)
print(f"{'COMPARATIVE METRICS — CART DECISION TREE (oversampled train)':^78}")
print("=" * 78)
header = (f"{'Split':<12} {'NTrain':>7} {'NTest':>7} "
         f"{'TrAcc':>7} {'TeAcc':>7} {'TrF1':>7} {'TeF1':>7} "
         f"{'TrAUC':>7} {'TeAUC':>7}")
print(header)
print("-" * 78)
for r in results:
    print(
        f"{r['name']:<12} {r['train_size']:>7,} {r['test_size']:>7,}"
        f" {r['train_acc']:>7.4f} {r['accuracy']:>7.4f}"
        f" {r['train_f1']:>7.4f} {r['f1']:>7.4f}"
        f" {r['train_auc']:>7.4f} {r['roc_auc']:>7.4f}"
    )
print("=" * 78)
print("Tr = Train (oversampled) · Te = Test (original distribution)")

## VII. Hyperparameter Search — Optimal `max_depth`

We iterate `max_depth` from 1 to 15 using the 80/20 split (with oversampled training) to find the point
where the model generalizes best without overfitting.

- **Underfitting** → very small `max_depth`: the tree fails to capture the wine chemistry complexity.
- **Overfitting** → very large `max_depth`: the tree memorizes the training set but fails on test.
- **Optimal** → where the test curve peaks and stabilizes.

In [ ]:
# ── CELL 8: Optimal max_depth search ──────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# Oversample TRAIN only for the depth search as well
X_tr_bal, y_tr_bal = oversample_minority(X_tr, y_tr, random_state=42)

depths = range(1, 16)
acc_train, acc_test = [], []
auc_train, auc_test = [], []
f1_train,  f1_test  = [], []

for d in depths:
    m = DecisionTreeClassifier(
        criterion='gini', max_depth=d,
        min_samples_split=10, min_samples_leaf=5,
        random_state=42
    )
    m.fit(X_tr_bal, y_tr_bal)
    # Train (balanced)
    acc_train.append(accuracy_score(y_tr_bal, m.predict(X_tr_bal)))
    auc_train.append(roc_auc_score(y_tr_bal, m.predict_proba(X_tr_bal)[:, 1]))
    f1_train.append(f1_score(y_tr_bal, m.predict(X_tr_bal), zero_division=0))
    # Test (original)
    acc_test.append(accuracy_score(y_te, m.predict(X_te)))
    auc_test.append(roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]))
    f1_test.append(f1_score(y_te, m.predict(X_te), zero_division=0))

best_depth = list(depths)[int(np.argmax(f1_test))]
print(f"🏆 Best max_depth by F1-Score (test): {best_depth}")
print(f"   Test F1-Score at depth={best_depth}: {max(f1_test):.4f}")

# ── Plot: train vs test learning curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Learning Curves — max_depth search\nCART Decision Tree — Wine Quality',
             fontsize=13, fontweight='bold')

metrics_plot = [
    ('Accuracy',  acc_train, acc_test),
    ('ROC-AUC',   auc_train, auc_test),
    ('F1-Score',  f1_train,  f1_test),
]
for ax, (title, train_vals, test_vals) in zip(axes, metrics_plot):
    ax.plot(depths, train_vals, 'o-', color='#1e3c72', lw=2, label='Train')
    ax.plot(depths, test_vals,  's-', color='#e67e22', lw=2, label='Test')
    ax.axvline(x=best_depth, color='#27ae60', linestyle='--', lw=1.5,
               label=f'Optimal: depth={best_depth}')
    ax.set_xlabel('max_depth', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.set_xticks(list(depths))

plt.tight_layout()
plt.show()
print("✅ max_depth curves generated")
print("   The gap between Train and Test curves is the visual signature of overfitting.")

## VIII. Visualizations

In [ ]:
# ── CELL 9: Comparative ROC curves ────────────────────────────────────────────
COLORS = ['#1e3c72', '#27ae60', '#e67e22']

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random classifier (AUC=0.50)')

for r, color in zip(results, COLORS):
    ax.plot(
        r['fpr'], r['tpr'], color=color, lw=2.2,
        label=f"{r['name']} — AUC = {r['roc_auc']:.4f}"
    )

ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('ROC Curves — CART Decision Tree\nPredicting High Quality Wine (quality >= 7)',
             fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()
print("✅ ROC curves generated")

In [ ]:
# ── CELL 10: Confusion Matrices (3 splits) ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices — CART Decision Tree — Wine Quality',
             fontsize=13, fontweight='bold')

for ax, r, color in zip(axes, results, COLORS):
    cm   = confusion_matrix(r['y_test'], r['y_pred'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Standard (0)', 'High quality (1)']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{r['name']}\nAccuracy={r['accuracy']:.4f}", fontsize=11)

plt.tight_layout()
plt.show()
print("✅ Confusion matrices generated")

In [ ]:
# ── CELL 11: Test metric comparison ───────────────────────────────────────────
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
labels       = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

x     = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (r, color) in enumerate(zip(results, COLORS)):
    vals = [r[m] for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=r['name'],
                  color=color, alpha=0.88)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim([0, 1.12])
ax.set_ylabel('Metric value (TEST)', fontsize=11)
ax.set_title('Test Metric Comparison — Three Splits\nCART Decision Tree — Wine Quality',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(y=1.0, color='#ccc', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()
print("✅ Comparative metrics chart generated")

## IX. Variable Importance — Feature Importance (Gini)

Unlike Logistic Regression (linear coefficients), the CART tree measures each variable's importance
through the **average Gini impurity reduction** across all nodes where the variable was used.

- A higher value means the variable contributes more to separating the classes.
- All importances sum to 1.0.

In [ ]:
# ── CELL 12: Variable importance (Gini Feature Importance) ────────────────────
ref_model    = results[2]['model']   # 80/20 split
importances  = pd.Series(ref_model.feature_importances_, index=X.columns)
importances_ord = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importances_ord.plot(kind='barh', ax=ax, color='#2a5298', alpha=0.85)
ax.set_xlabel('Gini importance — average impurity reduction', fontsize=11)
ax.set_title('Variable Importance — Feature Importance (Gini)\n'
             'CART Decision Tree (Split 80/20) — Wine Quality',
             fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)

for patch in ax.patches:
    width = patch.get_width()
    if width > 0.001:
        ax.text(width + 0.002, patch.get_y() + patch.get_height() / 2,
                f'{width:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Feature importance generated")
print(f"\n📊 Sum of all importances: {importances.sum():.6f}  (should be ≈ 1.0)")
print(f"\nTop 5 most important variables for classifying high quality:")
print(importances.sort_values(ascending=False).head(5).to_string())

## X. Final Metrics Table

In [ ]:
# ── CELL 13: Summary DataFrame of metrics (train + test) ──────────────────────
summary = pd.DataFrame([{
    'Split'      : r['name'],
    'N Train'    : r['train_size'],
    'N Test'     : r['test_size'],
    'Train Acc'  : round(r['train_acc'], 4),
    'Test Acc'   : round(r['accuracy'],  4),
    'Train F1'   : round(r['train_f1'],  4),
    'Test F1'    : round(r['f1'],        4),
    'Train AUC'  : round(r['train_auc'], 4),
    'Test AUC'   : round(r['roc_auc'],   4),
    'Precision'  : round(r['precision'], 4),
    'Recall'     : round(r['recall'],    4),
} for r in results])

summary.set_index('Split', inplace=True)

print("=" * 70)
print(f"{'FINAL METRICS TABLE (train + test)':^70}")
print("=" * 70)
try:
    display(summary.style
        .background_gradient(cmap='Greens',
                             subset=['Test Acc', 'Test F1', 'Test AUC', 'Precision', 'Recall'])
        .format('{:.4f}',
                subset=['Train Acc','Test Acc','Train F1','Test F1','Train AUC','Test AUC','Precision','Recall'])
        .set_caption("CART Decision Tree — Binary Classification: High Quality Wine (quality >= 7)")
    )
except:
    print(summary.to_string())

# Overfitting gap check
print("\n📌 Overfitting gap (Train F1 - Test F1):")
for r in results:
    gap = r['train_f1'] - r['f1']
    flag = "✅ healthy" if gap < 0.10 else ("⚠️ moderate" if gap < 0.20 else "❌ high")
    print(f"   {r['name']}: {gap:+.4f}  {flag}")

print(f"\n   Best max_depth found in section VII: {best_depth}")

## XI. Conclusions

### What did we learn with the CART tree?

| Aspect | Observation |
|---------|------------|
| **No scaling** | CART trees are invariant to scaling because they split on thresholds; this simplifies the pipeline vs Logistic Regression. |
| **Interpretability** | The tree can be visualized with `plot_tree`. Each node shows the split condition, the Gini impurity, and the class distribution. |
| **Feature Importance** | Unlike linear coefficients, Gini importance measures how much each variable reduces impurity on average. `alcohol` and `volatile acidity` are expected to be the most important, consistent with the EDA. |
| **Overfitting** | Deep trees memorize the training set. `max_depth` controls this trade-off; the optimal was selected in section VII. Reporting train+test metrics exposes the gap directly. |
| **Class imbalance** | Minority-class **oversampling with `resample`** (train only) lets the tree learn the minority class (high quality) without contaminating the test set. |
| **Stability** | If the three splits yield similar metrics, the model generalizes well regardless of the partition. |

### Comparison with Logistic Regression

| | Logistic Regression | CART Tree |
|---|---|---|
| **Scaling** | Required (StandardScaler) | Not required |
| **Decision boundary** | Linear | Non-linear (piecewise) |
| **Variable importance** | Coefficients (log-odds) | Gini impurity reduction |
| **Overfitting risk** | Low (implicit regularization) | High if max_depth is not controlled |
| **Interpretability** | Logistic equation | If-else rule tree |

### Suggested next steps
- Retrain the three splits using the `best_depth` found above.
- Apply cross-validation (k-fold) for a more reliable metric estimate.
- Evaluate whether changing the classification threshold (currently quality >= 7) to quality >= 6 improves class balance and metrics.